# Craft My Book

Uses the `llm` package in `src/` (OpenAI provider) to draft book content.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from llm import get_client, chat

In [2]:
client = get_client("openai")
model = "gpt-4o-mini"

In [3]:
prompt = "Write an engaging opening paragraph for a book about ..."

response = chat(client, model, prompt)
print(response)

Sure! Could you please provide me with a specific theme or subject for the book? That way, I can craft an engaging opening paragraph tailored to your needs!


## Module 1.2 — Speech Processing (Whisper)

Turns a lecture recording — video or audio — into a structured, timestamped,
domain-aware transcript (`src/ingestion/speech.py`, `src/ingestion/vocab.py`).
Requires the `ffmpeg` binary on PATH and `faster-whisper` installed
(`pip install -r requirements.txt`); the large-v3 model also wants a GPU for
reasonable speed.

Flow: `extract_vocab` (LLM pass over slide titles/filenames/headings — vocab is
per-corpus, never hardcoded) → `extract_audio` (ffmpeg; handles video or audio
sources) → `transcribe_audio` (faster-whisper, vocab-primed, VAD-filtered, word
timestamps, no cross-segment conditioning) → `clean_transcript` (conservative LLM
pass — fixes terms/punctuation, changes nothing else) → JSON.

In [ ]:
from ingestion import extract_audio, transcribe_audio, clean_transcript, process_source, extract_vocab, vocab_material_from_filenames, bootstrap_vocab_from_audio

# data/harvard-speech.wav: ~34s of real spoken English (public-domain Harvard sentences
# test corpus) - use this to smoke-test the pipeline locally before pointing it at a
# real lecture.
source_path = "data/harvard-speech.wav"

# Preferred: derive vocab from material that already exists around the recording
# (slide titles / filenames here; could also be PDF headings).
sibling_files: list[str] = []  # no slides for this sample -> falls through to bootstrap
vocab_material = vocab_material_from_filenames(sibling_files)
vocab = extract_vocab(vocab_material, client, model)

# Fallback: audio is the ONLY source (no slides/useful filenames) - bootstrap vocab from
# a fast draft transcription pass instead of skipping priming altogether.
if not vocab:
    audio_path = extract_audio(source_path)
    vocab = bootstrap_vocab_from_audio(audio_path, client, model)

vocab

In [ ]:
# Step by step (useful while inspecting intermediate output)
audio_path = extract_audio(source_path)  # works for video or audio sources
raw_transcript = transcribe_audio(audio_path, vocab=vocab)
cleaned_transcript = clean_transcript(raw_transcript, client, model)

print(f"{len(cleaned_transcript.segments)} segments, {cleaned_transcript.duration:.0f}s")
cleaned_transcript.segments[0]

In [ ]:
# Or, end to end in one call: source -> audio -> transcribe -> clean -> saved JSON
transcript = process_source(source_path, vocab=vocab, client=client, clean_model=model)
transcript.to_json  # already written to output/transcripts/lecture5.json by process_source